In [ ]:


import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100


database = 'saygexx'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))


# **Подзадача 1:**



 **1. Создаю свою БД в ClickHouse**

In [ ]:
client.command('''
    CREATE DATABASE IF NOT EXISTS saygexx ON CLUSTER '{cluster}';
''')

**2. Создаю распределённую таблицу в ClickHouse, которая разделена по шардам, реплицирована, хранит только актуальные данные - ClickHouse.**




In [ ]:
#Создание локальной таблицы
client.command('''CREATE TABLE IF NOT EXISTS saygexx.weather_local ON CLUSTER '{cluster}'
(
    `event_time` DateTime,
    `temperature_2m` Float64,
    `snowfall_mm` Float64,
    `load_date` Date,
    `city` String,
    `source` String,
    `imported_at` DateTime DEFAULT now()
)
ENGINE = ReplicatedMergeTree
PARTITION BY toYYYYMM(load_date)
ORDER BY (city, load_date, event_time) 
SETTINGS index_granularity = 8192;
''')

In [ ]:
# Создание таблицы реплицированной таблицы c актуальными данными
client.command('''
CREATE TABLE IF NOT EXISTS saygexx.weather_actual_distributed ON CLUSTER '{cluster}'
AS saygexx.weather_local
ENGINE = Distributed('{cluster}', 'saygexx', 'weather_local', cityHash64(city));
''')

In [ ]:
client.command('''INSERT INTO saygexx.weather_actual_distributed 
(event_time, temperature_2m, snowfall_mm, load_date, city, source)
VALUES
('2024-01-15 10:00:00', -5.3, 0.0, '2024-01-15', 'Perm', 'test'),
('2024-01-15 11:00:00', -4.8, 0.2, '2024-01-15', 'Perm', 'test');
''')

In [ ]:
#Чекаем данные
client.command('''SELECT* FROM saygexx.weather_actual_distributed  LIMIT 2''')

**3.Строю пайлайн: PG → Spark → трансформации → ClickHouse**

In [ ]:
#Открываю спарк сессию.
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("pg-to-clickhouse") \
    .config(
        "spark.jars.packages",
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0"
    ) \
    .getOrCreate()


In [ ]:
#Подключаюсь к постргресу с помощью спарк, читаю данные из таблички.
pg_df = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://postgres_source:5432/source")
    .option("dbtable", "public.order_events")
    .option("user", os.getenv("POSTGRES_USER"))
    .option("password", os.getenv("POSTGRES_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
)
pg_df.show(5)

In [ ]:
#Трансформируем данные
from pyspark.sql import functions as F

# 1. строим витрину
result_df = (
    pg_df
    .withColumnRenamed("id", "postgres_id")
    .withColumnRenamed("ts", "event_time")
    .withColumn("load_date", F.to_date("event_time"))
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn("source", F.lit("postgres"))
)

# 2. убираем миллисекунды
result_df = (
    result_df
    .withColumn("event_time", F.date_trunc("second", "event_time"))
    .withColumn("loaded_at", F.date_trunc("second", "loaded_at"))
)

result_df.show(5, False)


In [ ]:
#Опционально создаем бд если нет
# client.command("""
# CREATE DATABASE IF NOT EXISTS saygexx
# ON CLUSTER '{cluster}'
# """)


In [ ]:
#Создаем локальную таблицу в ClickHouse
client.command("""
CREATE TABLE IF NOT EXISTS saygexx.order_events_local ON CLUSTER '{cluster}'
(
    postgres_id Int32,
    order_id Int32,
    status String,
    event_time DateTime,
    load_date Date,
    loaded_at DateTime,
    source String
)
ENGINE = ReplicatedMergeTree(
    '/clickhouse/tables/{shard}/saygexx/order_events_local',
    '{replica}'
)
PARTITION BY toYYYYMM(load_date)
ORDER BY (order_id, event_time)
""")


In [ ]:
#Cоздаем Distributed таблицу для актуальных данных в ClickHouse
client.command("""
CREATE TABLE IF NOT EXISTS saygexx.order_events ON CLUSTER '{cluster}'
AS saygexx.order_events_local
ENGINE = Distributed(
    '{cluster}',
    'saygexx',
    'order_events_local',
    rand()
)
""")


In [ ]:
# Вариант если таблица записывается впервые
# (
#     result_df.write
#     .format("jdbc")
#     .option("url", "jdbc:clickhouse://clickhouse01:8123/default")
#     .option("dbtable", "saygexx.order_events")
#     .option("user", os.getenv("CLICKHOUSE_USER"))
#     .option("password", os.getenv("CLICKHOUSE_PASSWORD"))
#     .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
#     .mode("append")
#     .save()
# )
# result_df.show(5,False)

In [ ]:
#Если нам надо обновить данные из таблички постгресс и перенести их в клик используем этот вариант.
# max_ts = client.query(
#     "SELECT max(event_time) FROM saygexx.order_events"
# ).result_rows[0][0]

# print("Последняя загруженная дата:", max_ts)
# pg_df = (
#     spark.read
#     .format("jdbc")
#     .option("url", "jdbc:postgresql://postgres_source:5432/source")
#     .option("dbtable", f"(SELECT * FROM public.order_events WHERE ts > '{max_ts}') as t")
#     .option("user", os.getenv("POSTGRES_USER"))
#     .option("password", os.getenv("POSTGRES_PASSWORD"))
#     .option("driver", "org.postgresql.Driver")
#     .load()
# )


In [ ]:
# считаем количество строк в источнике PostgreSQL
pg_count = (
    spark.read                     # говорим: будем ЧИТАТЬ данные
    .format("jdbc")                # источник чтения — JDBC (подключение к БД)
    
    # адрес базы данных
    .option("url", "jdbc:postgresql://postgres_source:5432/source")
    
    # какую таблицу читаем
    .option("dbtable", "public.order_events")
    
    # логин
    .option("user", os.getenv("POSTGRES_USER"))
    
    # пароль
    .option("password", os.getenv("POSTGRES_PASSWORD"))
    
    # jdbc драйвер для postgres
    .option("driver", "org.postgresql.Driver")
    
    # выполняем чтение → получаем DataFrame
    .load()
    
    # action → запускается реальное выполнение задачи
    .count()
)

# печатаем результат
print("PG rows:", pg_count)





In [ ]:
# сколько в ClickHouse
ch_count = client.query(
    "SELECT count() FROM saygexx.order_events"
).result_rows[0][0]

print("PG:", pg_count)
print("CH:", ch_count)

if pg_count == ch_count:
    print("✅ Всё загрузилось корректно")
else:
    print("❌ Потеря данных!")

In [ ]:
spark.stop()

# **Подзаадача 2:**

1) Необходимо содать таблицу заказов  создать с движком `Log`

| Поле         | Описание                            |
|--------------|-------------------------------------|
| `order_id`   | Уникальный ID заказа                |
| `user_id`    | ID пользователя                     |
| `order_date` | Дата и время оформления заказа      |
| `total_amount` | Сумма заказа                      |
| `paid`       |  Признак оплаты: оплачено, не оплачено|
| `items`      |  Список ID товаров в заказе         |

2) Выбрать  подходящие типы данных для колонок
3) После создания вставь подходящие строки
4) Вывести выборку строк **

In [ ]:
#Подключаемся к базе данных
import clickhouse_connect
import pandas as pd
import os

pd.options.display.max_colwidth = 100


database = 'saygexx'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

In [ ]:
client.command('''
    CREATE DATABASE IF NOT EXISTS saygexx ON CLUSTER '{cluster}';
''')

In [ ]:
client.command('''
CREATE TABLE IF NOT EXISTS saygexx.orders_task2 (
    order_id UUID,
    user_id UInt32,
    order_date DateTime,
    total_amount Decimal64(2),
    paid UInt8 DEFAULT 0,
    items Array(UInt32)
) ENGINE = Log
''')

In [ ]:
import uuid
from datetime import datetime
#uuid.uuid4() генерирует уникальный id
rows = [
    (str(uuid.uuid4()), 3001, datetime.now(), 100.50, 1, [1,2]),
    (str(uuid.uuid4()), 3002, datetime.now(), 200.00, 0, [5]),
    (str(uuid.uuid4()), 3003, datetime.now(), 999.99, 1, [7,8,9]),
]
#Вставка данных
client.insert(
    "saygexx.orders_task2",
    rows,
    column_names=["order_id", "user_id", "order_date", "total_amount", "paid", "items"]
)


In [ ]:
# #Проверка данных в таблице
# client.query("SELECT count() FROM saygexx.orders_task2").result_rows

client.query_df(f'''
    SELECT * FROM  {database}.orders_task2
''')

1) Выполни запросы в следующей ячейке.
2) Сделай так чтобы UNION ALL выполнился
3) Типы данных, которые не могут быть автоматически преобразованы системой должны быть, как у таблицы 1. В случае коализий преобразования типов данных, необходимо ставить значение NULL.

In [ ]:
# client.command(f'''
#     DROP TABLE IF EXISTS {database}.table_1;
# ''')

client.command(f'''
CREATE TABLE  {database}.table_1
(
    order_id     UInt32, 
    user_id      UInt16,  
    total_amount Float32,   
    paid         Nullable(UInt8),     
    item_count   Int8   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_1 VALUES (10001, 321, 1499.50, 1, 3), (10001,  321, 1499.50 , NULL, 3), (10001, 321, , 1, 3);
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.table_2;
''')

client.command(f'''
CREATE TABLE  {database}.table_2
(
    order_id     Int64, 
    user_id      UInt32,  
    total_amount Nullable(Decimal(5, 2)),   
    paid         UInt8,     
    item_count   String   
)
ENGINE = Log;
''')

client.command(f'''
    INSERT INTO {database}.table_2 VALUES (10001, 123456, 899.99, 0, '5'), (10001, 123456, NULL, 0, '5'), (10001, 123456, 899.99, 0, '5 тут специально вписаны буквы') ;
''')

In [ ]:
# Исправлять ошибку
client.query_df(f'''
    SELECT order_id, user_id, total_amount, paid, item_count FROM {database}.table_1
    UNION ALL
    SELECT order_id, user_id, toFloat32(total_amount),paid,toInt8OrNull(item_count) FROM {database}.table_2
''')

### Самостоятельная работа на проверку работоспостобности TTL:

1) Создать таблицу с движком MergeTree c 2мя колонками: id Int32, dt DateTime
2) Указать только сотировку по id. PRIMARY KEY указывать не нужно оно будет совпадать по умолчанию с ORDER BY
3) Написать в условии TTL удаление строки по dt через 10 секунд
4) вставиить данные через конструкцию `INSERT INTO <имя БД>.<имя таблицы> AS SELECT number, now() + number FROM numbers(100)` (`numbers(100)` это таблица которая генерирует колонку number со значениями от 0 до 99)
5) выполнить select твоей таблицы и посмотреть сколько в ней строк
6) 'Удаление происходит после слияния поэтому выполнить команду' ` OPTIMIZE TABLE <имя БД>.<имя таблицы> FINAL;` через 20 секунд после выполнения шага 5
7) выполнить повторно шаг 5
8) теперь вопрорять 5-6 шаги пока в таблице совсем не останется данных

In [ ]:
client.command('''
CREATE TABLE IF NOT EXISTS saygexx.ttl_work(
                id UInt32,
                dt dateTime
)                
ENGINE = MergeTree()
ORDER BY (id)
TTL dt + INTERVAL 10 second DELETE
''') 
               
            

In [ ]:
client.command('''
              INSERT INTO saygexx.ttl_work 
              SELECT number,now() + number FROM numbers(100)
'''            
)

In [ ]:
client.command ('''
                SELECT count(*) 
                FROM saygexx.ttl_work
'''                
 )

In [ ]:
client.command('''OPTIMIZE TABLE saygexx.ttl_work FINAL;
''')

### Самостоятельная работа на расшиненные атрибуты колонок:

1) Создать таблицу `product_sales`, которая будет хранить информацию о продажах товаров. В таблице необходимо использовать **все указанные выше атрибуты хотя бы по одному разу**.

| Поле          | Тип данных  | Атрибуты                                                              |
|---------------|-------------|------------------------------------------------------------------------|
| `sale_id`     | `UInt64`    | `COMMENT`: "Уникальный идентификатор продажи"                        |
| `price`       | `Float32`   | `DEFAULT`: 0.0                                                        |
| `quantity`    | `UInt8`     | `DEFAULT`: 1                                                          |
| `total`       | `Float32`   | `MATERIALIZED`: `price * quantity`                                   |
| `description` | `String`    | `CODEC`: `ZSTD(1)`                                                    |
| `taxed_total` | `Float32`   | `ALIAS`: `total * 1.2`
2) Вставить пару строк в данную таблицу данные в данную таблицу
3) Вывести данные через колонки и через `*`.
4) Попробуй вставить данные в колонки `total` и `taxed_total`. Посмотреть результат

In [ ]:
client.command('''
         CREATE TABLE saygexx.product_sales
         (sale_id UInt64 COMMENT 'Иднетификатор продажи',
         price Float32 DEFAULT 0.0,
         quantity UInt8 DEFAULT 1,
         total Float32 MATERIALIZED price * quantity,
         description String CODEC(ZSTD(10)),
         taxed_total Float32 ALIAS total * 1.2
         )
         ENGINE = MergeTree()
         ORDER BY sale_id
''')         

In [ ]:
#Перед вставкой данных можно посмотреть в какие колонки можно вставлять
client.command('''DESCRIBE TABLE saygexx.product_sales
'''
)


materialized хранится физически, alias вычисляется во время чтения

In [ ]:
client.command('''INSERT INTO saygexx.product_sales
               (sale_id,price,quantity,description)
                VALUES(2,12.5,2,'Cвежий Арбуз')
                (3,7.0,4,'Тухлый апельсин');
''')

In [ ]:
client.query_df(f'''
    SELECT 
        sale_id
      , price 
      , quantity 
      , total 
      , description
      , taxed_total
    FROM {database}.product_sales
''')

In [ ]:
# total и taxed_total не отбразятся если их не указать напрямую
client.query_df(f'''
    SELECT 
        *
    FROM {database}.product_sales
''')

# **Атрибуты при создании колонок**

Значения по умолчанию в ClickHouse

| Тип    | default    |
| ------ | ---------- |
| числа  | 0          |
| String | ''         |
| Date   | 1970-01-01 |
| Bool   | 0          |

| Тип        | Значение            |
| ---------- | ------------------- |
| UInt / Int | 0                   |
| Float      | 0.0                 |
| String     | ''                  |
| Date       | 1970-01-01          |
| DateTime   | 1970-01-01 00:00:00 |

| тип                | не передали колонку → что будет |
| ------------------ | ------------------------------- |
| `UInt8`            | 0                               |
| `String`           | ""                              |
| `Date`             | 1970-01-01                      |
| `Nullable(UInt8)`  | NULL                            |
| `Nullable(String)` |                                 |


In [ ]:
# client.command(f'''
#     DROP TABLE IF EXISTS {database}.nl_lc_tabl;
# ''')

client.command(f'''
    CREATE TABLE {database}.nl_lc_tabl (
        a Nullable(UInt32),         -- разрешает вставку с пропуском значения.
        b LowCardinality(String),   -- ускоряет работу малокоординальных данных(часто повторяющихся)
        c UInt32
    ) ENGINE = MergeTree 
    ORDER BY tuple(); -- определяет порядок сток по порядку вставки данных
''')

client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (NULL,'test' ,1); -- null вставится корректно
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1,'test2',NULL); -- null вставится как 0.
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, NULL, 3); -- null вставляется как пустая строка
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, 'test2', 4);
''')

In [ ]:
client.query_df(f'''
    SELECT 
        a, 
        b, 
        c 
    FROM {database}.nl_lc_tabl
''')

In [ ]:

client.command(f'''
    SET input_format_null_as_default = 0; -- параметр отвечающий за вставку пустых значений. Выполняется совместно с командой на вставку
''')
# появится ошибка при вставке
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (2, 'sdasda', NULL)
''')


In [ ]:
client.query_df(f'''
    SELECT 
        toTypeName(a), 
        toTypeName(b), 
        toTypeName(c) 
    FROM {database}.nl_lc_tabl
''')

### Самостоятельная работа на партиционирование таблиц:
1) Создайть таблицу `customer_orders` с партиционированием по месяцу `PARTITION BY toYYYYMM(order_date)` и первичным ключом сортирвки  `ORDER BY` - `customer_id, order_date`. Поле `total` должно быть `MATERIALIZED`. 

| Поле             | Тип         | Описание                                 |
|------------------|-------------|------------------------------------------|
| `order_id`       | UInt64      | Уникальный идентификатор заказа          |
| `order_date`     | DateTime    | Дата и время заказа                      |
| `customer_id`    | UInt32      | ID клиента                               |
| `product_name`   | String      | Название товара                          |
| `quantity`       | UInt8       | Кол-во единиц                            |
| `price`          | Float32     | Цена за единицу                          |
| `total`          | Float32     | MATERIALIZED: `quantity * price`         |

2) Вставьте данные за несколько разных месяцев по  (можно использовать `now() - INTERVAL X DAY`)
4) Выведи данные на экран с помощью `*` и убедись в какой парции нахаодятся твои данные добавив скрытую колонку `_part`

In [ ]:
client.command('''
               CREATE TABLE IF NOT EXISTS saygexx.customer_orders(
               order_id UInt64,
               order_date DateTime,
               customer_id UInt32,
               product_name String,
               quantity UInt8,
               price Float32,
               total Float32 MATERIALIZED quantity * price     
            )         
            Engine = MergeTree
            PARTITION BY toYYYYMM(order_date)
            ORDER BY (customer_id,order_date); 
'''
)                    
                     
 
 

In [ ]:
client.command('''
INSERT INTO saygexx.customer_orders
SELECT
     number AS order_id,
     now() - INTERVAL (number % 90) DAY AS order_date, -- даст разные мясяцы
     number % 10 AS customer_id,
     concat('product_',toString(number)) AS product_name,
     (number % 5) + 1 AS quantity,
     100 AS price
From numbers(30);     

'''
)


In [ ]:
client.command('''SELECT*,_part
            FROM saygexx.customer_orders
'''            
            )

### Самостоятельная работа на комбинаторы:

1) Выполни запрос запрос, чтобы создать тестовую таблицу
2) Напиши 3 запроса отвечающие на следующие вопросы:
- Сколько уникальных пользователей заходили с **мобильных устройств**
- Среднее время просмотра только на странице `home`
- Для каждого устройства — количество уникальных пользователей
- Собери список страниц, на которые заходил пользователь в индефикатором `2`, только с десктопа (`web`)

**Подсказака**:
В запросах нужно исполльзовать следующие компненты: `groupArray`, `If`, `avg`, `uniq`

In [ ]:
client.command(f'''
    CREATE TABLE {database}.page_views
    (
        user_id UInt32,
        page String,
        device String,
        view_time UInt8
    )
    ENGINE = MergeTree
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.page_views (user_id, page, device, view_time) VALUES
        (1, 'home', 'mobile', 10),
        (1, 'catalog', 'mobile', 15),
        (2, 'home', 'web', 8),
        (2, 'checkout', 'web', 20),
        (2, 'home', 'mobile', 5),
        (4, 'catalog', 'web', 12),
        (5, 'checkout', 'mobile', 7),
        (6, 'home', 'web', 9),
        (7, 'home', 'mobile', 11),
        (8, 'catalog', 'web', 13);
''')

In [ ]:
client.command('''SELECT 
                uniqIf(user_id, device = 'mobile'),
                avgIf(view_time, page = 'home'),
                uniq(user_id),
                groupArrayIf(page, user_id = 2 AND device = 'web')
          FROM saygexx.page_views
'''
)                


                

In [ ]:
client.query_df(f'''
    SELECT
        user_id,
        groupArrayIf(page, device = 'web') AS web_pages_visited
    FROM {database}.page_views
    WHERE user_id = 2
    GROUP BY user_id    
''')

In [ ]:
client.query_df(f'''
    SELECT 
        device, 
        uniq(user_id) 
    FROM {database}.page_views
    GROUP BY device
''')

#### Агрегаторные типы данных

Сущесвует 2 агригаторных типа данных:
- SimpleAggregateFunction - предназначет для хранения простых агрегатов, которое хранит конечно состояние
- AggregateFunction - сложные агрегаты, которая хранит состояние всех добавленных значений